# 🔬 02 — Clustering Experiments

**Responsibility:** Apply and compare clustering algorithms (KMeans and DBSCAN)
on electric vehicle datasets to identify groups of punctual and unpunctual users,
and assign them a standard deviation (STD).

**Main outputs:**
- `vehicles_clustered.csv` — Dataset with basic KMeans clustering (3 clusters)
- `vehicles_punctuality_clusters.csv` — Binary clustering (2 groups) with STD
- `dbscan_eps_results.csv` / `data_kmeans.csv` — Comparative experiment results


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix

## 2. Feature Engineering

Reusable function to calculate the **derived metrics** used across all
clustering experiments. These features capture charging behaviour
and the driver's 'stress' level.


In [ ]:
def aplicar_feature_engineering(df):
    """
    Calculates derived features on an electric vehicle DataFrame.

    Computed features:
    - battery_level: current normalized charge level [0, 1]
    - window_length: length of the availability window (hours)
    - required_energy: energy needed for a full charge (kWh)
    - required_time: time needed for a full charge (hours)
    - stress_index: ratio required_time / available_window.
                    Values > 1 indicate there is not enough time to charge
    - time_of_day: start time slot (night/morning/afternoon/evening)
    """

    df = df.copy()

    df["battery_level"] = df["current_charge"] / df["battery_capacity"]
    df["window_length"] = df["available_time_end"] - df["available_time_start"]
    df["required_energy"] = df["battery_capacity"] - df["current_charge"]
    df["required_time"] = df["required_energy"] / df["charge_speed"]
    # Replace window_length=0 with 0.5 to avoid division by zero
    df["stress_index"] = df["required_time"] / df["window_length"].replace(0, 0.5)

    def map_time(hour):
        """Converts the numeric hour into a categorical time slot."""
        if 0 <= hour < 6:   return "night"
        elif 6 <= hour < 12: return "morning"
        elif 12 <= hour < 18: return "afternoon"
        else:                 return "evening"

    df["time_of_day"] = df["available_time_start"].apply(map_time)

    return df


def preparar_features_para_clustering(df):
    """
    Preprocesses the selected features for clustering:
    - Selects relevant columns
    - Applies OneHotEncoding to categorical variables
    - Normalizes with StandardScaler

    Returns:
        X_scaled: numpy array with normalized features
    """

    # Features to use: mix of numeric and categorical
    features = ["driver_age", "driver_profession", "use_frequency",
                "battery_level", "stress_index", "time_of_day", "origin_distance"]

    X = df[features].copy()

    # OneHot encoding for categorical variables (drop='first' avoids multicollinearity)
    encoder = OneHotEncoder(drop="first", sparse_output=False)
    categorical = X[["driver_profession", "time_of_day"]]
    encoded = encoder.fit_transform(categorical)

    # Numeric variables without transformation (only normalized later)
    numeric = X.drop(columns=["driver_profession", "time_of_day"]).values

    # Combine numeric and encoded variables
    X_proc = np.hstack([numeric, encoded])

    # Normalization: mean=0, std=1 — essential for KMeans and DBSCAN
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_proc)

    return X_scaled


## 3. Experiment 1: KMeans with 3 Clusters and Heuristic STD Assignment

Groups drivers into 3 clusters and assigns the STD (punctuality uncertainty)
using heuristic rules based on the mean characteristics of each cluster.


In [ ]:
# --- Load and prepare the base dataset ---
df = pd.read_csv("vehicles_extended.csv")
df = aplicar_feature_engineering(df)
X_scaled = preparar_features_para_clustering(df)

# --- KMeans with 3 clusters ---
# n_clusters=3 → three punctuality levels (high, medium, low)
# random_state=42 → reproducibility
# n_init=10 → repeats 10 times with different initializations and keeps the best
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# --- Heuristic STD assignment per cluster ---
# STD represents uncertainty in punctuality:
# low STD → predictable user, high STD → unpredictable user
std_mapping = {}

for cluster in df["cluster"].unique():
    cluster_data = df[df["cluster"] == cluster]
    avg_freq = cluster_data["use_frequency"].mean()
    avg_dist = cluster_data["origin_distance"].mean()
    avg_stress = cluster_data["stress_index"].mean()

    # Heuristic rules:
    # - High frequency + low stress + nearby origin → stable user (low STD)
    # - Low frequency OR high stress OR distant origin → uncertain user (high STD)
    if avg_freq > 4 and avg_stress < 1.0 and avg_dist < 15:
        std = 0.25   # very predictable
    elif avg_freq < 3 or avg_stress > 2.0 or avg_dist > 20:
        std = 1.0    # very uncertain
    else:
        std = 0.5    # medium variability

    std_mapping[cluster] = std

df["assigned_std"] = df["cluster"].map(std_mapping)

# --- Save results ---
df.to_csv("vehicles_clustered.csv", index=False)
print("✅ Clustering KMeans (3 grupos) completado")
print("STD asignada por cluster:", std_mapping)


## 4. Experiment 2: Binary KMeans (Punctual vs Unpunctual)

Clustering into **2 groups** for binary punctual/unpunctual classification.
Uses a weighted punctuality score to assign interpretable labels.


In [ ]:
# --- Load the precision level 1 dataset (hardest mix) ---
df = pd.read_csv("vehicles_precision_1.csv")
df = aplicar_feature_engineering(df)

# --- Compute composite punctuality score ---
# Combines stress_index with risk factors based on profession and age

# Profession risk: how much it increases the probability of being unpunctual
profession_risk = {"student": 1.2, "unemployed": 1.3, "worker": 1.0, "retired": 0.9}

# Age risk: younger drivers are slightly less predictable
# Normalized between 0.8 (older) and 1.2 (younger)
age_risk = 1.2 - 0.4 * (df["driver_age"] - df["driver_age"].min()) / \
           (df["driver_age"].max() - df["driver_age"].min())

# Final score: stress_index carries more weight (60%) than demographic factors
df["punctuality_score"] = (
    df["stress_index"] * 0.6 +
    age_risk * 0.2 +
    df["driver_profession"].map(profession_risk) * 0.2
)

# Base label using the median as separation threshold
median_score = df["punctuality_score"].median()
df["user_type"] = np.where(df["punctuality_score"] > median_score, "unpunctual", "punctual")

X_scaled = preparar_features_para_clustering(df)

# --- KMeans with 2 clusters ---
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# --- Assign label to cluster by majority ---
# The cluster with the most 'punctual' members is labeled as punctual
cluster_map = {}
for c in df["cluster"].unique():
    majority_type = df[df["cluster"] == c]["user_type"].mode()[0]
    cluster_map[c] = majority_type
df["cluster_label"] = df["cluster"].map(cluster_map)

# --- Fixed STD per interpretable group ---
std_mapping = {"punctual": 0.25, "unpunctual": 0.75}
df["assigned_std"] = df["cluster_label"].map(std_mapping)

df.to_csv("vehicles_punctuality_clusters.csv", index=False)
print("✅ Clustering binario (puntual vs impuntual) completado")
print("Distribución:", df["cluster_label"].value_counts().to_dict())
print("STD asignada:", std_mapping)


## 5. Experiment 3: KMeans Configuration Comparison (Silhouette Score)

Evaluates different KMeans configurations to find the one that produces
the most compact and well-separated clusters, measured by the **Silhouette Score**
(the closer to 1, the better the separation).


In [ ]:
from sklearn.preprocessing import LabelEncoder

# --- Load and prepare the data ---
data = pd.read_csv("vehicles_precision_2.csv")

# Label encoding (simpler than OneHot for this experiment)
le = LabelEncoder()
data["driver_profession"] = le.fit_transform(data["driver_profession"])

features = [
    "distance", "available_time_start", "available_time_end", "current_charge",
    "battery_capacity", "charge_speed", "discharge_rate",
    "driver_age", "driver_profession", "use_frequency", "origin_distance"
]
X = data[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Experiment definitions ---
# Each experiment varies: number of initializations, max iterations and init method
# - 'k-means++': smart initialization (more stable, converges faster)
# - 'random': random initialization (more variable, may get stuck in local minima)
experiments = [
    {"name": "Very basic (worst)",      "n_init": 1,   "max_iter": 100, "init": "random",    "random_state": None},
    {"name": "Basic stable",            "n_init": 10,  "max_iter": 200, "init": "k-means++", "random_state": 42},
    {"name": "Medium stability",        "n_init": 20,  "max_iter": 300, "init": "k-means++", "random_state": 42},
    {"name": "Refined",                 "n_init": 50,  "max_iter": 400, "init": "k-means++", "random_state": 42},
    {"name": "High precision",          "n_init": 100, "max_iter": 500, "init": "k-means++", "random_state": 42},
    {"name": "Exploratory (variable)",  "n_init": 30,  "max_iter": 300, "init": "random",    "random_state": 0},
]

results = []

# --- Run each experiment and record the Silhouette Score ---
for exp in experiments:
    kmeans = KMeans(
        n_clusters=2,
        n_init=exp["n_init"],
        max_iter=exp["max_iter"],
        init=exp["init"],
        random_state=exp["random_state"]
    )
    clusters = kmeans.fit_predict(X_scaled)
    # Silhouette Score: measures how well-separated the clusters are
    # Range [-1, 1]: values close to 1 = well-defined clusters
    score = silhouette_score(X_scaled, clusters)
    results.append((exp["name"], score))
    print(f'{exp["name"]}: Silhouette Score = {score:.4f}')

# --- Show the best configuration ---
best = max(results, key=lambda x: x[1])
print("\n✅ Mejor configuración:", best[0], "con Silhouette Score =", round(best[1], 4))


## 6. Experiment 4: DBSCAN — Density-Based Clustering

DBSCAN detects clusters naturally without requiring K to be specified.
Points that do not belong to any dense cluster are labeled as **noise (-1)**,
allowing atypical users to be identified.


In [ ]:
# --- Load and prepare data ---
df = pd.read_csv("vehicles_extended.csv")
df = aplicar_feature_engineering(df)

# Compute punctuality score (same as in experiment 2)
profession_risk = {"student": 1.2, "unemployed": 1.3, "worker": 1.0, "retired": 0.9}
age_risk = 1.2 - 0.4 * (df["driver_age"] - df["driver_age"].min()) / \
           (df["driver_age"].max() - df["driver_age"].min())
df["punctuality_score"] = (
    df["stress_index"] * 0.6 + age_risk * 0.2 +
    df["driver_profession"].map(profession_risk) * 0.2
)
median_score = df["punctuality_score"].median()
df["user_type"] = np.where(df["punctuality_score"] > median_score, "unpunctual", "punctual")

X_scaled = preparar_features_para_clustering(df)

# --- DBSCAN ---
# eps=0.6: maximum radius for two points to be neighbors
#          (smaller value = more compact clusters, more points as noise)
# min_samples=8: minimum number of neighbors for a point to be a cluster "core"
#                (larger value = fewer clusters, more tolerant of noise)
dbscan = DBSCAN(eps=0.6, min_samples=8)
df["cluster"] = dbscan.fit_predict(X_scaled)

# --- Map clusters to punctuality labels ---
# Cluster -1 is noise (atypical users, not classifiable)
cluster_map = {}
for c in df["cluster"].unique():
    if c == -1:
        cluster_map[c] = "noise"  # points with no cluster → treated with high STD
    else:
        majority_type = df[df["cluster"] == c]["user_type"].mode()[0]
        cluster_map[c] = majority_type
df["cluster_label"] = df["cluster"].map(cluster_map)

# --- Differentiated STD: noise has the highest uncertainty ---
std_mapping = {"punctual": 0.25, "unpunctual": 0.75, "noise": 0.5}
df["assigned_std"] = df["cluster_label"].map(std_mapping)

# --- Save results ---
df.to_csv("vehicles_punctuality_clusters_dbscan.csv", index=False)

print("✅ DBSCAN completado")
print("Distribución de clusters:", df["cluster"].value_counts().to_dict())
print("(Nota: cluster -1 = ruido/outliers)")


## 7. Experiment 5: DBSCAN Hyperparameter Grid Search

Searches for the best combination of `eps` and `min_samples` by maximizing the **Adjusted Rand Index (ARI)**,
which measures how well the clustering matches the real punctuality labels.


In [ ]:
# --- Load evaluation dataset (with known real labels) ---
df = pd.read_csv("dataset_final.csv")
y_true = df["punctuality_group"]
X = df.drop(columns=["punctuality_group"])

# --- Preprocessing with ColumnTransformer ---
cat_cols = [col for col in X.columns if X[col].dtype == "object"]
num_cols = [col for col in X.columns if X[col].dtype != "object"]

preprocessor = ColumnTransformer([
    ("onehot", OneHotEncoder(drop="first"), cat_cols),
    ("scale",  StandardScaler(), num_cols)
])
X_processed = preprocessor.fit_transform(X)

# PCA for visualization (not for clustering) — reduce to 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

# --- Grid Search: explore combinations of eps and min_samples ---
eps_values = np.arange(1.0, 2.0, 0.1)       # from 1.0 to 1.9
min_samples_values = [4, 5, 6, 7, 8]         # minimum neighborhood size

results = []

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=float(eps), min_samples=int(min_samples))
        clusters = dbscan.fit_predict(X_processed)

        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        n_noise = list(clusters).count(-1)
        # ARI: compares cluster assignment with real labels
        # 1.0 = perfect match, 0.0 = random, negative = worse than chance
        ari = adjusted_rand_score(y_true, clusters)

        results.append({
            "eps": round(float(eps), 2),
            "min_samples": int(min_samples),
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "ARI": round(ari, 4)
        })

# --- Show the top 10 results ---
results_df = pd.DataFrame(results).sort_values("ARI", ascending=False)
print("Top 10 combinaciones de parámetros por ARI:")
print(results_df.head(10))

# --- Save grid search results ---
results_df.to_csv("dbscan_grid_search_results.csv", index=False)
print("\n✅ Grid search completado y guardado")


## 8. Experiment 6: Logistic Metrics Simulation (Cost and Timespan)

Simulates how the **total charging cost** and **total time** vary as a function of
the percentage of unpunctual users, noise level and problem size.
Generates CSV files that will be used in the visualization notebook.


In [ ]:
def simulate_metrics_logic(df_sim, algo_prefix):
    """
    Calculates simulated logistic metrics from the standard deviations
    assigned by the clustering algorithm.

    Logic:
    - Higher uncertainty (high STD) → higher planning cost
    - Critical errors (predicting punctual when user is unpunctual) → additional penalty

    Returns:
        (cost, timespan): tuple with estimated total cost and duration
    """
    std_col = f'std_{algo_prefix}'
    label_col = f'label_{algo_prefix}'

    total_uncertainty = df_sim[std_col].sum()
    # Critical error: algorithm says punctual but user is unpunctual
    # This type of error is the most costly in charge planning
    critical_errors = df_sim[
        (df_sim['punctuality_group'] == 'late') & (df_sim[label_col] == 'punctual')
    ].shape[0]

    cost = 95 + (total_uncertainty * 0.75) + (critical_errors * 4)
    timespan = 10 + (total_uncertainty * 0.08)

    return cost, timespan


def run_simulation_to_file(algo_name, output_filename):
    """
    Runs a battery of simulations for a given algorithm and saves the
    results in CSV for later visualization.

    Experiment dimensions:
    - Problem sizes: [20, 40, 60, 80, 100] vehicles
    - Noise levels: [5%, 15%, 30%]
    - Punctual/unpunctual balances: [90/10, 50/50, 10/90]
    - Per combination: 5 repetitions to smooth out randomness
    """

    problem_sizes = [20, 40, 60, 80, 100]
    noise_levels = [0.05, 0.15, 0.30]
    balances = [(0.9, 0.1), (0.5, 0.5), (0.1, 0.9)]

    results = []

    for noise in noise_levels:
        for balance in balances:
            for size in problem_sizes:
                temp_c, temp_t = [], []

                # Average 5 runs to reduce stochastic variance
                for _ in range(5):
                    punctual_count = int(size * balance[0])
                    raw_data = []

                    for i in range(size):
                        is_punctual = i < punctual_count
                        is_noisy = np.random.random() < noise
                        # Nearby origin = punctual; distant = unpunctual
                        o_dist = 5 if (is_punctual and not is_noisy) else 30
                        dist = max(0, np.random.normal(o_dist, 2))
                        raw_data.append({
                            "dist": dist, "prof": 0 if o_dist == 5 else 1,
                            "punctuality_group": "punctual" if is_punctual else "late"
                        })

                    df_sim = pd.DataFrame(raw_data)
                    X_scaled = StandardScaler().fit_transform(df_sim[['dist', 'prof']])

                    if algo_name == "dbscan":
                        db = DBSCAN(eps=1.5, min_samples=3).fit(X_scaled)
                        df_sim['cluster'] = db.labels_
                        valid_c = df_sim[df_sim['cluster'] != -1]
                        p_clust = valid_c.groupby('cluster')['dist'].mean().idxmin() \
                                  if len(valid_c) > 0 else -99
                        df_sim['label_dbscan'] = df_sim['cluster'].apply(
                            lambda x: 'punctual' if x == p_clust else 'late')
                        df_sim['std_dbscan'] = df_sim['cluster'].apply(
                            lambda x: 0.5 if x == p_clust else (3.0 if x == -1 else 2.0))
                        c, t = simulate_metrics_logic(df_sim, "dbscan")
                    else:  # kmeans
                        km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_scaled)
                        df_sim['cluster'] = km.labels_
                        p_clust = df_sim.groupby('cluster')['dist'].mean().idxmin()
                        df_sim['label_kmeans'] = df_sim['cluster'].apply(
                            lambda x: 'punctual' if x == p_clust else 'late')
                        df_sim['std_kmeans'] = df_sim['cluster'].apply(
                            lambda x: 0.5 if x == p_clust else 2.0)
                        c, t = simulate_metrics_logic(df_sim, "kmeans")

                    temp_c.append(c)
                    temp_t.append(t)

                results.append({
                    "noise": noise, "balance": balance[0], "size": size,
                    "cost": np.mean(temp_c), "timespan": np.mean(temp_t)
                })

    pd.DataFrame(results).to_csv(output_filename, index=False)
    print(f"✅ Simulación '{algo_name}' guardada en: {output_filename}")


# --- Run simulations for both algorithms ---
run_simulation_to_file("kmeans", "data_kmeans.csv")
run_simulation_to_file("dbscan", "data_dbscan.csv")
print("\n✅ Todos los experimentos completados. Archivos listos para visualización.")
